In [1]:
%pip install torch

Note: you may need to restart the kernel to use updated packages.


In [3]:
import torch
import torch.nn as nn
import torch.quantization as tq
from transformers import AutoModelForCausalLM , AutoTokenizer

In [29]:
model_name="distilgpt2"

model=AutoModelForCausalLM.from_pretrained(model_name)
tokenizer=AutoTokenizer.from_pretrained(model_name,use_fast=False)

c:\Users\INMOR14\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [31]:
qat_config=tq.QConfig(
    activation=tq.FakeQuantize.with_args(
        observer=tq.MovingAverageMinMaxObserver,
        quant_min=0,quant_max=255,
        dtype=torch.quint8,qscheme=torch.per_tensor_affine
    ),
    weight=tq.FakeQuantize.with_args(
        observer=tq.MinMaxObserver,
        quant_min=-128,quant_max=27,
        dtype=torch.qint8,qscheme=torch.per_tensor_symmetric
    ),

)

In [32]:
for name,module in model.named_modules():
    if isinstance(module,nn.Embedding):
        module.qconfig=None



In [ ]:
#attach quant/dequant stubs
model.qconfig=qat_config

In [ ]:
model.train()

#model now has quant/dequant ops
tq.prepare_qat(model,inplace=True)

C:\Users\INMOR14\AppData\Local\Temp\ipykernel_30376\1746117857.py:2: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  tq.prepare_qat(model,inplace=True)


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm(
          (768,), eps=1e-05, elementwise_affine=True
          (activation_post_process): FakeQuantize(
            fake_quant_enabled=tensor([1], dtype=torch.uint8), observer_enabled=tensor([1], dtype=torch.uint8), quant_min=0, quant_max=255, dtype=torch.quint8, qscheme=torch.per_tensor_affine, ch_axis=-1, scale=tensor([1.]), zero_point=tensor([0], dtype=torch.int32)
            (activation_post_process): MovingAverageMinMaxObserver(min_val=inf, max_val=-inf)
          )
        )
        (attn): GPT2Attention(
          (c_attn): Conv1D()
          (c_proj): Conv1D()
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm(
          (768,), eps=1e-05, elementwise_affin

In [35]:
inputs=tokenizer("Quantization Aware Training on LLMs!",return_tensors="pt")

In [36]:
labels=inputs["input_ids"]

In [37]:
optimizer=torch.optim.AdamW(model.parameters(),lr=5e-5)

In [39]:
for step in range(50):
    output=model(**inputs,labels=labels)
    loss=output.loss
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    if step%10==0:
        print(f"Step {step} | loss: {loss:.4f}")

Step 0 | loss: 9.3698
Step 10 | loss: 2.5159
Step 20 | loss: 1.1978
Step 30 | loss: 0.7103
Step 40 | loss: 0.0060


In [40]:
qat_model=tq.convert(model.eval(),inplace=False)

C:\Users\INMOR14\AppData\Local\Temp\ipykernel_30376\4124601729.py:1: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  qat_model=tq.convert(model.eval(),inplace=False)
